In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import seaborn as sns


# ---------------------------------------------------------
# Fallback Solver: Fit GJR-GARCH(1,1,1) via Scipy MLE
# ---------------------------------------------------------
def fit_gjr_garch_scipy(returns):
  scale = 100.0
  r = returns.values * scale
  mu = np.mean(r)
  eps = r - mu
  var_eps = np.var(eps)

  def neg_log_lik(params):
    omega, alpha, gamma, beta = params
    if (
        omega <= 0
        or alpha < 0
        or gamma < 0
        or beta < 0
        or (alpha + 0.5 * gamma + beta) >= 1.0
    ):
      return 1e10
    n = len(eps)
    sigma2 = np.zeros(n)
    sigma2[0] = var_eps
    for t in range(1, n):
      e_prev = eps[t - 1]
      s_prev = sigma2[t - 1]
      indicator = 1.0 if e_prev < 0 else 0.0
      sigma2[t] = (
          omega
          + alpha * (e_prev**2)
          + gamma * indicator * (e_prev**2)
          + beta * s_prev
      )
      if sigma2[t] <= 0:
        sigma2[t] = 1e-6
    return 0.5 * np.sum(np.log(2 * np.pi * sigma2) + (eps**2) / sigma2)

  bounds = [(1e-6, 1.0), (1e-6, 0.49), (1e-6, 0.49), (1e-6, 0.99)]
  constraints = {'type': 'ineq', 'fun': lambda p: 0.999 - (p[1] + 0.5 * p[2] + p[3])}

  init_p = [var_eps * 0.05, 0.05, 0.10, 0.80]
  res = minimize(
      neg_log_lik,
      init_p,
      method='SLSQP',
      bounds=bounds,
      constraints=constraints,
  )

  omega_s, alpha, gamma, beta = res.x
  n = len(eps)
  sigma2_s = np.zeros(n)
  sigma2_s[0] = var_eps
  for t in range(1, n):
    e_prev = eps[t - 1]
    s_prev = sigma2_s[t - 1]
    indicator = 1.0 if e_prev < 0 else 0.0
    sigma2_s[t] = (
        omega_s
        + alpha * (e_prev**2)
        + gamma * indicator * (e_prev**2)
        + beta * s_prev
    )

  omega = omega_s / (scale**2)
  cond_vol = pd.Series(np.sqrt(sigma2_s) / scale, index=returns.index)

  eps_orig = returns.values - (mu / scale)
  std_residuals = pd.Series(eps_orig / cond_vol.values, index=returns.index)

  indicator_t = (eps_orig < 0).astype(float)
  forecast_var = (
      omega
      + alpha * (eps_orig**2)
      + gamma * indicator_t * (eps_orig**2)
      + beta * (cond_vol.values**2)
  )
  forecast_vol = pd.Series(np.sqrt(forecast_var), index=returns.index)

  return {
      'mu': mu / scale,
      'omega': omega,
      'alpha': alpha,
      'gamma': gamma,
      'beta': beta,
      'cond_vol': cond_vol,
      'std_residuals': std_residuals,
      'forecast_vol': forecast_vol,
  }


def run_phase_7():
  print('==================================================')
  print('     RUNNING PHASE 7: GJR-GARCH VOLATILITY SCALING')
  print('==================================================')

  # ---------------------------------------------------------
  # 1. Load Input Data
  # ---------------------------------------------------------
  input_file = 'portfolio_phase3_output(10).csv'
  if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"'{input_file}' not found! Please run Phase 3 first."
    )

  df_p3 = pd.read_csv(input_file, parse_dates=['Date'], index_col='Date')
  returns = df_p3['Daily_Return']
  pnl = df_p3['Daily_PnL']
  dates = df_p3.index

  window = 250
  conf_level = 0.99
  alpha_var = 1.0 - conf_level

  # ---------------------------------------------------------
  # 2. Fit GJR-GARCH(1,1,1) Model
  # ---------------------------------------------------------
  print(' Fitting GJR-GARCH model on portfolio returns...')

  try:
    from arch import arch_model

    am = arch_model(returns * 100.0, vol='Garch', p=1, o=1, q=1, dist='normal')
    res = am.fit(disp='off')

    mu = res.params['mu'] / 100.0
    omega = res.params['omega'] / (100.0**2)
    alpha = res.params['alpha[1]']
    gamma = res.params['gamma[1]']
    beta = res.params['beta[1]']

    cond_vol = res.conditional_volatility / 100.0
    std_residuals = res.std_resid

    eps = (returns - mu).values
    indicator_t = (eps < 0).astype(float)
    forecast_var = (
        omega + alpha * (eps**2) + gamma * indicator_t * (eps**2) + beta * (cond_vol**2)
    )
    forecast_vol = pd.Series(np.sqrt(forecast_var), index=dates)

    print(' [✓] Fitted via arch package.')

  except ImportError:
    print(' [!] arch package not found. Using internal Scipy MLE solver...')
    fit_res = fit_gjr_garch_scipy(returns)
    mu = fit_res['mu']
    omega = fit_res['omega']
    alpha = fit_res['alpha']
    gamma = fit_res['gamma']
    beta = fit_res['beta']
    cond_vol = fit_res['cond_vol']
    std_residuals = fit_res['std_residuals']
    forecast_vol = fit_res['forecast_vol']
    print(' [✓] Fitted via Scipy MLE.')

  print(
      f'     Params -> Omega: {omega:.8f} | Alpha: {alpha:.4f} | Gamma'
      f' (Leverage): {gamma:.4f} | Beta: {beta:.4f}'
  )
  print(
      f'     Persistence (Alpha + 0.5*Gamma + Beta):'
      f' {alpha + 0.5*gamma + beta:.4f}'
  )

  # ---------------------------------------------------------
  # 3. Scale Returns & Calculate GJR-GARCH VaR
  # ---------------------------------------------------------
  raw_var_series = pd.Series(index=dates, dtype=float)
  gjr_var_series = pd.Series(index=dates, dtype=float)

  print(' Calculating rolling GJR-GARCH-scaled Historical VaR...')
  for i in range(window, len(dates)):
    curr_forecast = forecast_vol.iloc[i]

    win_pnl = pnl.iloc[i - window : i]
    win_vols = cond_vol.iloc[i - window : i]

    # Raw Historical VaR
    raw_var = -np.percentile(win_pnl, alpha_var * 100)
    raw_var_series.iloc[i] = raw_var

    # GJR-GARCH Scaled Historical VaR
    scaled_pnl = win_pnl * (curr_forecast / win_vols)
    gjr_var = -np.percentile(scaled_pnl, alpha_var * 100)
    gjr_var_series.iloc[i] = gjr_var

  # ---------------------------------------------------------
  # 4. Save Outputs to CSV
  # ---------------------------------------------------------
  summary_df = pd.DataFrame(
      {
          'Conditional_Volatility': cond_vol,
          'Forecast_Volatility': forecast_vol,
          'Standardized_Residuals': std_residuals,
          'Raw_99_VaR': raw_var_series,
          'GJR_GARCH_Scaled_99_VaR': gjr_var_series,
      },
      index=dates,
  )
  summary_df.to_csv('gjr_garch_scaled_var_summary_phase7(10).csv')
  print(' [✓] Saved Data -> gjr_garch_scaled_var_summary_phase7(10).csv')

  # ---------------------------------------------------------
  # 5. Generate & Save Charts
  # ---------------------------------------------------------
  sns.set_theme(style='whitegrid')

  # CHART 1: GJR-GARCH Conditional Volatility
  plt.figure(figsize=(12, 5))
  plt.plot(
      cond_vol.index,
      cond_vol * np.sqrt(252),
      color='darkred',
      lw=1.5,
      label='Annualized GJR-GARCH Volatility (Asymmetric)',
  )
  plt.title('Chart 1: GJR-GARCH Annualized Conditional Volatility')
  plt.xlabel('Date')
  plt.ylabel('Annualized Volatility (%)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart1_gjr_garch_volatility(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart1_gjr_garch_volatility(10).png')

  # CHART 2: Standard GARCH vs. GJR-GARCH Volatility Comparison
  plt.figure(figsize=(12, 5))
  if os.path.exists('garch_scaled_var_summary_phase6.csv'):
    p6_df = pd.read_csv(
        'garch_scaled_var_summary_phase6.csv',
        parse_dates=['Date'],
        index_col='Date',
    )
    plt.plot(
        p6_df.index,
        p6_df['Conditional_Volatility'] * np.sqrt(252),
        color='navy',
        lw=1.2,
        alpha=0.8,
        label='Standard GARCH(1,1) Volatility (Symmetric)',
    )

  plt.plot(
      cond_vol.index,
      cond_vol * np.sqrt(252),
      color='darkred',
      lw=1.2,
      alpha=0.8,
      linestyle='--',
      label='GJR-GARCH(1,1,1) Volatility (Asymmetric)',
  )
  plt.title('Chart 2: Standard GARCH vs. GJR-GARCH Conditional Volatility')
  plt.xlabel('Date')
  plt.ylabel('Annualized Volatility (%)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart2_garch_vs_gjr_garch_volatility(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart2_garch_vs_gjr_garch_volatility(10).png')

  # CHART 3: Leverage Effect Visualization (News Impact Curve)
  z_range = np.linspace(-4, 4, 200)
  avg_vol = np.mean(cond_vol)
  eps_range = z_range * avg_vol

  news_impact_var = (
      omega
      + alpha * (eps_range**2)
      + gamma * (eps_range < 0) * (eps_range**2)
      + beta * (avg_vol**2)
  )
  news_impact_vol = np.sqrt(news_impact_var) * np.sqrt(252)

  plt.figure(figsize=(10, 5))
  plt.plot(
      z_range,
      news_impact_vol,
      color='crimson',
      lw=2.2,
      label=f'News Impact Curve (γ = {gamma:.4f})',
  )
  plt.axvline(0, color='gray', linestyle='--', lw=1)
  plt.title('Chart 3: Leverage Effect Visualization (News Impact Curve)')
  plt.xlabel('Return Shock z_t (Standard Deviations)')
  plt.ylabel('Next-Day Annualized Volatility (%)')
  plt.annotate(
      'Asymmetric Spike\n(Negative Shock / Bad News)',
      xy=(-2.5, np.max(news_impact_vol) * 0.95),
      color='darkred',
      fontweight='bold',
  )
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart3_leverage_effect_visualization(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart3_leverage_effect_visualization(10).png')

  # CHART 4: Historical VaR vs. GJR-GARCH-Scaled VaR
  plt.figure(figsize=(12, 6))
  plt.plot(
      raw_var_series.index,
      raw_var_series,
      label='Raw 99% Historical VaR ($)',
      color='darkorange',
      lw=1.5,
      linestyle='--',
  )
  plt.plot(
      gjr_var_series.index,
      gjr_var_series,
      label='GJR-GARCH-Scaled 99% Historical VaR ($)',
      color='darkred',
      lw=1.8,
  )
  plt.title(
      'Chart 4: Rolling 250-Day Historical VaR vs. GJR-GARCH-Scaled VaR'
  )
  plt.xlabel('Date')
  plt.ylabel('1-Day Potential Loss ($)')
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart4_historical_vs_gjr_garch_var(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart4_historical_vs_gjr_garch_var(10).png')

  print('\n==================================================')
  print('          PHASE 7 COMPLETED SUCCESSFULLY!         ')
  print('==================================================')


if __name__ == '__main__':
  run_phase_7()